In [ ]:
# ======================================================================
# ✨ AI 코딩 튜터의 오늘의 실습! ✨
# 데이터셋: lemon-mint/korean_parallel_sentences_v1.1
# 주제: 기계 번역(Machine Translation) 원리 맛보기
# 목표: 전 세계의 언어 데이터를 파이썬으로 다뤄보는 경험을 쌓아봅시다!
# ======================================================================

# 이 데이터셋은 한국어와 영어의 '병렬 문장(Parallel Sentences)'을 담고 있어요.
# 즉, 영어 문장과 그에 대응하는 한국어 문장 쌍이 수십만 개나 있는 보물창고랍니다!
# 우리는 이 데이터를 이용해 '번역 모델'이 어떻게 작동할지 재미있게 탐색해 볼 거예요.

import random
from datasets import load_dataset
from tqdm import tqdm # 진행 상황을 보기 좋게 만들어주는 마법의 도구입니다!

# -------------------------
# ⚙️ 설정 및 데이터 로드 (Curating the Dataset)
# -------------------------

DATASET_NAME = "lemon-mint/korean_parallel_sentences_v1.1"
SAMPLE_COUNT = 50 # 전체 데이터셋이 너무 크니, 딱 50개 샘플만 가지고 놀아봅시다!

print(f"🌈 [튜터 모드] 데이터셋 '{DATASET_NAME}' 로딩을 시작합니다...")

dataset = None
try:
    # 1. 스트리밍 모드 시도 (가장 빠르고 메모리 효율적입니다!)
    print("✅ 1단계: Streaming 모드로 데이터셋을 불러오려고 합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 스트리밍 모드로 데이터를 가져올 준비가 완료되었어요. 메모리 걱정 NO!")

except Exception as e:
    # 스트리밍 모드가 실패할 경우, 안전하게 소량만 다운로드하여 진행합니다.
    print(f"⚠️ 경고! 스트리밍 로딩에 문제가 생겨 fallback 모드로 전환합니다. ({e})")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("📚 소량 다운로드 완료! 안전하게 다음 실습을 진행할 수 있습니다.")
    except Exception as e_fallback:
        print(f"😭😭 치명적인 오류 발생: 데이터셋 로드에 실패했습니다. ({e_fallback})")
        exit()


# -------------------------
# 🛠️ 데이터 샘플링 (Sampling the Data)
# -------------------------

# 주의: 스트리밍 모드에서는 전체 길이를 알 수 없기 때문에, .take()를 사용해
# 메모리 효율적으로 앞의 N개 데이터만 뽑아와요.
if hasattr(dataset, "take"):
    # 스트리밍 모드일 경우, .take() 메소드를 사용해서 샘플만 가져옵니다.
    print(f"\n➡️ 2단계: 상위 {SAMPLE_COUNT}개 샘플만 추출합니다...")
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
else:
    # 일반 Dataset 객체라면, list()로 변환하는 것이 가장 안전합니다.
    sampled_dataset = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

print(f"✅ 데이터 준비 완료! 총 {len(sampled_dataset)}개의 샘플을 분석할 거예요.")

# ======================================================================
# 🚀 미션 1: 데이터 구조 파악하기 (Simple Exploration)
# ======================================================================
print("\n" + "="*80)
print("🎯 미션 1: 데이터의 심장을 들여다보기 (데이터 샘플 확인)")
print("=====================================================================")

print("💡 Tip: 이 데이터셋의 핵심은 'korean'과 'english' 두 개의 문자열 필드를 가진다는 것입니다.")

print("\n--- [첫 번째 샘플] ---")
sample = sampled_dataset[0]
print(f"🇰🇷 한국어 (Korean): {sample['korean'][:40]}...")
print(f"🇬🇧 영어 (English): {sample['english'][:40]}...")

print("\n--- [세 번째 샘플] ---")
sample = sampled_dataset[2]
print(f"🇰🇷 한국어 (Korean): {sample['korean'][:40]}...")
print(f"🇬🇧 영어 (English): {sample['english'][:40]}...")

# ======================================================================
# 📈 미션 2: 정량적 분석 (Quantitative Analysis - 길이 비교)
# ======================================================================
print("\n" + "="*80)
print("🚀 미션 2: 번역 스타일 분석 (글자 수 길이 비교)")
print("=====================================================================")
print("🔍 분석 목표: 영어 문장과 한국어 문장의 글자 수(character count)가 어떤 경향을 보이는지 분석해 봅시다.")

korean_lengths = []
english_lengths = []

# 모든 샘플을 순회하며 길이를 계산합니다.
for sample in sampled_dataset:
    korean_lengths.append(len(sample['korean']))
    english_lengths.append(len(sample['english']))

# 평균과 합계를 계산합니다.
avg_korean = sum(korean_lengths) / len(korean_lengths)
avg_english = sum(english_lengths) / len(english_lengths)

# 길이 차이의 평균을 구합니다.
avg_diff = avg_korean - avg_english

print(f"\n📝 {len(sampled_dataset)}개 샘플 분석 결과:")
print(f"   - 🇰🇷 한국어 평균 글자 수: {avg_korean:.2f} 글자")
print(f"   - 🇬🇧 영어 평균 글자 수: {avg_english:.2f} 글자")
print(f"   - 📊 평균 길이 차이 (한-영): {avg_diff:.2f} 글자 (🇰🇷이 더 길까요? 🇬🇧이 더 길까요?)")

if avg_diff > 0:
    print("👉 튜터 코멘트: 한국어 쪽의 정보량이 조금 더 많은 편인 것 같네요! 흥미롭지 않나요?")
elif avg_diff < 0:
    print("👉 튜터 코멘트: 영어 쪽의 길이가 상대적으로 더 길군요. 번역 문화의 차이를 느낄 수 있어요!")
else:
    print("👉 튜터 코멘트: 아주 균형 잡힌 데이터를 가진 것 같습니다!")


# ======================================================================
# 🤖 미션 3: 창의적 AI 사용 예시 (Prompt Engineering 시뮬레이션)
# ======================================================================
print("\n" + "="*80)
print("🧠 미션 3: AI에게 질문 던지기 (Prompt Generation Simulation)")
print("=====================================================================")
print("💬 목표: 만약 이 데이터를 LLM(거대 언어 모델)에 넣고 특정 작업을 시키려면 어떤 '지시문(Prompt)'을 만들어야 할까요?")

def generate_prompt_task(korean_text: str, english_text: str) -> str:
    """
    두 개의 병렬 문장 쌍을 입력받아, LLM에게 최적화된 질문(프롬프트)을 생성하는 함수입니다.
    """
    print(f"\n--- [작업 시뮬레이션] ---")
    
    # 초보자도 쉽게 이해할 수 있도록, 주어진 문장의 핵심을 뽑아내게 요청하는 프롬프트 패턴을 만듭니다.
    prompt = f"""
    [원본 한국어 문장]: "{korean_text}"
    [원본 영어 문장]: "{english_text}"

    위 두 문장을 참고하여, 이 문장이 설명하는 '가장 핵심적인 개념'을 10자 이내의 키워드로 요약해 주세요.
    요약만 출력하고 다른 설명은 붙이지 마세요.
    """
    return prompt

# 무작위로 3개의 샘플을 골라 미션을 수행합니다.
random_samples = random.sample(sampled_dataset, min(3, len(sampled_dataset)))

for i, sample in enumerate(random_samples):
    prompt_output = generate_prompt_task(sample['korean'], sample['english'])
    print(f"\n✨ [샘플 {i+1}에 대한 가상의 LLM 프롬프트]:")
    print("--------------------------------------------------")
    print(prompt_output)

print("\n=====================================================================")
print("🎉 축하합니다! 🎉")
print("여러분이 데이터 로드, 전처리, 정량적 분석, 그리고 창의적인 활용(프롬프트 설계)까지 완벽하게 경험했습니다.")
print("파이썬으로 언어 데이터를 다루는 것은 정말 재미있답니다! 꾸준히 연습해요! 파이팅! 💪")
print("=====================================================================")